# Project 20: Warehouse Inventory & Stockout Risk Analysis
**Industry:** Supply Chain / Logistics  |  **Tools:** Python, NumPy, Pandas

**Business problem:** A warehouse wants to know which products may run out soon (stockout risk), which products are sitting unsold for too long (overstock), how much money is tied up in stock, and which suppliers are slow or have many defects.

**Note:** The dataset is *synthetic* (made for teaching). The results describe this dataset only. They show patterns, not proof of cause and effect.

## Step 0 - Import libraries and load settings

In [1]:
import numpy as np
import pandas as pd
import os

# File locations (the notebook, data folder and outputs folder sit together)
root = "."
data_file = root + "/data/20_warehouse_inventory.csv"
output_folder = root + "/outputs"
os.makedirs(output_folder, exist_ok=True)

# Show wider tables and normal numbers (no scientific notation)
pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 30)
pd.options.display.float_format = "{:,.2f}".format

# Business rules (we can change them here)
overstock_days = 90        # more than 90 days of stock = overstock
print("Ready. Data file is:", data_file)

Ready. Data file is: ./data/20_warehouse_inventory.csv


## Task 1 - Load and inspect the dataset

In [2]:
df = pd.read_csv(data_file)

print("Rows and columns:", df.shape)
print()
print("Column names:")
print(list(df.columns))
print()
print("Data types:")
print(df.dtypes)
print()
print("First 5 rows:")
print(df.head())

Rows and columns: (1212, 11)

Column names:
['SKU_ID', 'Product_Name', 'Category', 'Supplier', 'Warehouse_Location', 'Current_Stock', 'Reorder_Level', 'Avg_Daily_Sales', 'Unit_Cost', 'Lead_Time_Days', 'Defect_Return_Rate_Pct']

Data types:
SKU_ID                        str
Product_Name                  str
Category                      str
Supplier                      str
Warehouse_Location            str
Current_Stock               int64
Reorder_Level               int64
Avg_Daily_Sales           float64
Unit_Cost                 float64
Lead_Time_Days            float64
Defect_Return_Rate_Pct    float64
dtype: object

First 5 rows:
    SKU_ID       Product_Name        Category          Supplier Warehouse_Location  Current_Stock  Reorder_Level  Avg_Daily_Sales  Unit_Cost  \
0  SKU1199          Spice Box         Grocery   FastTrack Goods      Bengaluru Hub            184             62             5.20     509.74   
1  SKU2062             Hoodie         Apparel      Apex Traders      

## Task 2 - Missing values and duplicate SKU_ID records
Each SKU should appear only once. If a SKU appears twice, its stock and value would be counted twice.

In [3]:
print("Missing values in each column:")
print(df.isnull().sum())
print()

# Rows whose SKU_ID appears more than once
duplicate_rows = df[df.duplicated(subset="SKU_ID", keep=False)]
print("Rows with a repeated SKU_ID:", len(duplicate_rows))
print(duplicate_rows.sort_values("SKU_ID"))

Missing values in each column:
SKU_ID                     0
Product_Name               0
Category                   0
Supplier                   0
Warehouse_Location         0
Current_Stock              0
Reorder_Level              0
Avg_Daily_Sales            0
Unit_Cost                 26
Lead_Time_Days            32
Defect_Return_Rate_Pct     0
dtype: int64

Rows with a repeated SKU_ID: 24
       SKU_ID       Product_Name     Category          Supplier Warehouse_Location  Current_Stock  Reorder_Level  Avg_Daily_Sales  Unit_Cost  \
977   SKU1047       Sports Socks      Apparel      Apex Traders        Kolkata Hub           2811            157            14.50     185.70   
450   SKU1047       Sports Socks      Apparel      Apex Traders        Kolkata Hub           2811            157            14.50     185.70   
314   SKU1129     Cotton T-Shirt      Apparel   FastTrack Goods      Delhi-NCR Hub            307             92             8.70   1,066.04   
315   SKU1129     Cotton T-S

In [4]:
# The repeated rows are exact copies, so we keep the first one and delete the others
rows_before = len(df)
df = df.drop_duplicates(subset="SKU_ID", keep="first")
df = df.reset_index(drop=True)

print("Rows before:", rows_before)
print("Rows after :", len(df))
print("Are all SKU_IDs unique now?", df["SKU_ID"].is_unique)

Rows before: 1212
Rows after : 1200
Are all SKU_IDs unique now? True


## Task 3 - Clean missing Lead_Time_Days and Unit_Cost
**Method I chose**

* **Lead_Time_Days:** fill with the **median lead time of the same supplier**. A supplier usually delivers in a similar time, and the median is not affected by a few very long delays.
* **Unit_Cost:** fill with the **median unit cost of the same category**. Prices differ a lot between categories, and the median is not pulled by a few very expensive items.

I keep a True/False column so I can always see which values were filled in. Limitation: filled values are only estimates.

In [5]:
# Remember which rows were missing
df["Lead_Time_Was_Missing"] = df["Lead_Time_Days"].isnull()
df["Unit_Cost_Was_Missing"] = df["Unit_Cost"].isnull()

# Fill lead time with the median of the same supplier
for supplier_name in df["Supplier"].unique():
    is_this_supplier = df["Supplier"] == supplier_name
    median_lead = df.loc[is_this_supplier, "Lead_Time_Days"].median()
    needs_fill = is_this_supplier & df["Lead_Time_Days"].isnull()
    df.loc[needs_fill, "Lead_Time_Days"] = median_lead

# Fill unit cost with the median of the same category
for category_name in df["Category"].unique():
    is_this_category = df["Category"] == category_name
    median_cost = df.loc[is_this_category, "Unit_Cost"].median()
    needs_fill = is_this_category & df["Unit_Cost"].isnull()
    df.loc[needs_fill, "Unit_Cost"] = median_cost

print("Lead times filled:", df["Lead_Time_Was_Missing"].sum())
print("Unit costs filled:", df["Unit_Cost_Was_Missing"].sum())
print("Missing values left:", df[["Lead_Time_Days", "Unit_Cost"]].isnull().sum().sum())

Lead times filled: 32
Unit costs filled: 26
Missing values left: 0


## Task 4 - Inventory value of every SKU and overall
Inventory value = Current_Stock x Unit_Cost

In [6]:
df["Inventory_Value"] = df["Current_Stock"] * df["Unit_Cost"]

total_value = df["Inventory_Value"].sum()
print("Number of SKUs          :", len(df))
print("Total units in stock    :", df["Current_Stock"].sum())
print("Total inventory value   :", round(total_value, 2))
print("Average value per SKU   :", round(df["Inventory_Value"].mean(), 2))
print("Median value per SKU    :", round(df["Inventory_Value"].median(), 2))

Number of SKUs          : 1200
Total units in stock    : 879746
Total inventory value   : 731770314.14
Average value per SKU   : 609808.6
Median value per SKU    : 156989.99


## Task 5 - SKUs at stockout risk
A SKU is at risk when `Current_Stock <= Reorder_Level`.

In [7]:
df["Stockout_Risk"] = df["Current_Stock"] <= df["Reorder_Level"]
df["Units_Below_Reorder"] = df["Reorder_Level"] - df["Current_Stock"]
df.loc[df["Units_Below_Reorder"] < 0, "Units_Below_Reorder"] = 0

risk_skus = df[df["Stockout_Risk"] == True]
risk_skus = risk_skus.sort_values("Units_Below_Reorder", ascending=False)

risk_count = len(risk_skus)
print("SKUs at stockout risk:", risk_count, "out of", len(df))
print("Percentage at risk   :", round(risk_count / len(df) * 100, 1), "%")
print()
print("15 SKUs with the biggest shortage:")
show_columns = ["SKU_ID", "Product_Name", "Category", "Current_Stock", "Reorder_Level", "Units_Below_Reorder"]
print(risk_skus[show_columns].head(15))

risk_skus.to_csv(output_folder + "/stockout_risk_skus.csv", index=False)

SKUs at stockout risk: 233 out of 1200
Percentage at risk   : 19.4 %

15 SKUs with the biggest shortage:
       SKU_ID    Product_Name        Category  Current_Stock  Reorder_Level  Units_Below_Reorder
669   SKU1805      Power Bank     Electronics            172            813                  641
30    SKU1670    Dumbbell Set          Sports            127            614                  487
1164  SKU1275      Pulses 1kg         Grocery            257            664                  407
113   SKU1104   Notebook Pack      Stationery            134            526                  392
946   SKU2198      Pulses 1kg         Grocery             37            421                  384
197   SKU1592      Power Bank     Electronics             67            421                  354
521   SKU2098     Storage Jar  Home & Kitchen             43            388                  345
488   SKU1555     File Folder      Stationery            242            570                  328
728   SKU1048  Desk Or

## Task 6 - Stockout-risk percentage by category

In [8]:
total_per_category = df.groupby("Category")["SKU_ID"].count()
risk_per_category = risk_skus.groupby("Category")["SKU_ID"].count()

risk_table = pd.DataFrame()
risk_table["Total_SKUs"] = total_per_category
risk_table["At_Risk_SKUs"] = risk_per_category
risk_table["Risk_Percent"] = risk_table["At_Risk_SKUs"] / risk_table["Total_SKUs"] * 100
risk_table = risk_table.sort_values("Risk_Percent", ascending=False)

print(risk_table.round(1))

                Total_SKUs  At_Risk_SKUs  Risk_Percent
Category                                              
Electronics            239            52         21.80
Home & Kitchen         252            53         21.00
Grocery                187            38         20.30
Apparel                209            40         19.10
Stationery             153            25         16.30
Sports                 160            25         15.60


## Task 7 - Rank suppliers by lead time and defect/return rate
For both measures a **lower number is better**, so rank 1 = best. The overall rank adds the two ranks together (both count equally - this is my simple choice).

In [9]:
supplier_table = df.groupby("Supplier")[["Lead_Time_Days", "Defect_Return_Rate_Pct"]].mean()
supplier_table["SKU_Count"] = df.groupby("Supplier")["SKU_ID"].count()

supplier_table["Lead_Time_Rank"] = supplier_table["Lead_Time_Days"].rank(method="min")
supplier_table["Defect_Rank"] = supplier_table["Defect_Return_Rate_Pct"].rank(method="min")
supplier_table["Combined_Score"] = supplier_table["Lead_Time_Rank"] + supplier_table["Defect_Rank"]
supplier_table["Overall_Rank"] = supplier_table["Combined_Score"].rank(method="min")

supplier_table = supplier_table.sort_values(["Overall_Rank", "Lead_Time_Days"])
print(supplier_table.round(2))

supplier_table.to_csv(output_folder + "/supplier_ranking.csv")

                   Lead_Time_Days  Defect_Return_Rate_Pct  SKU_Count  Lead_Time_Rank  Defect_Rank  Combined_Score  Overall_Rank
Supplier                                                                                                                       
FastTrack Goods              5.75                    2.16        181            1.00         3.00            4.00          1.00
Apex Traders                 6.49                    1.73        196            2.00         2.00            4.00          1.00
Delta Components             6.93                    1.15        162            3.00         1.00            4.00          1.00
BlueLine Supplies            8.87                    2.57        183            4.00         4.00            8.00          4.00
CityWave Pvt Ltd            12.07                    3.41        183            5.00         6.00           11.00          5.00
GreenField Co               14.64                    3.00        157            6.00         5.00       

## Task 8 - Top 10 highest-value SKUs

In [10]:
top10 = df.sort_values("Inventory_Value", ascending=False).head(10)
print(top10[["SKU_ID", "Product_Name", "Category", "Current_Stock", "Unit_Cost", "Inventory_Value"]])

top10_percent = top10["Inventory_Value"].sum() / total_value * 100
print()
print("These 10 SKUs hold", round(top10_percent, 1), "% of the total inventory value.")

       SKU_ID       Product_Name        Category  Current_Stock  Unit_Cost  Inventory_Value
214   SKU1615           Football          Sports           7107   3,044.02    21,633,850.14
360   SKU1853  Bluetooth Speaker     Electronics           4819   2,783.42    13,413,300.98
930   SKU1464        Storage Jar  Home & Kitchen           4096   3,012.57    12,339,486.72
645   SKU1684           Football          Sports           7776   1,168.81     9,088,666.56
1101  SKU1796     Cotton T-Shirt         Apparel          12949     695.63     9,007,712.87
317   SKU1479       Cookware Set  Home & Kitchen           4175   2,036.17     8,501,009.75
1090  SKU1134  Bluetooth Speaker     Electronics           1960   4,200.60     8,233,176.00
1011  SKU1456             Webcam     Electronics           3480   2,317.24     8,063,995.20
941   SKU1336           Keyboard     Electronics           3321   2,319.68     7,703,657.28
78    SKU1441             Webcam     Electronics           6110   1,195.52     7

## Task 9 - Days of stock remaining
Days of stock = Current_Stock / Avg_Daily_Sales

**Zero sales:** dividing by 0 is a problem. So I first replace 0 sales with a "missing" value (`NaN`). Then the result is also "missing" ("no sales, cannot estimate") instead of infinity.

In [11]:
safe_sales = df["Avg_Daily_Sales"].replace(0, np.nan)
df["Days_Of_Stock"] = df["Current_Stock"] / safe_sales

zero_sales_count = (df["Avg_Daily_Sales"] == 0).sum()
print("SKUs with zero daily sales:", zero_sales_count)
print("Any infinite values?", np.isinf(df["Days_Of_Stock"]).any())
print()
print(df["Days_Of_Stock"].describe().round(2))

SKUs with zero daily sales: 47
Any infinite values? False

count   1,153.00
mean       63.20
std        70.03
min         0.00
25%        22.92
50%        37.01
75%        56.55
max       285.79
Name: Days_Of_Stock, dtype: float64


## Task 10 - Slow-moving / overstocked items
A SKU is **overstocked** if it has more than 90 days of stock, **or** it has stock but **no sales at all**.

In [12]:
too_many_days = df["Days_Of_Stock"] > overstock_days
no_sales_but_stock = (df["Avg_Daily_Sales"] == 0) & (df["Current_Stock"] > 0)
df["Overstock_Flag"] = too_many_days | no_sales_but_stock

overstock = df[df["Overstock_Flag"] == True]
overstock_value = overstock["Inventory_Value"].sum()

print("Overstocked SKUs:", len(overstock), "(", round(len(overstock) / len(df) * 100, 1), "% of SKUs )")
print("Value tied up   :", round(overstock_value, 2), "(", round(overstock_value / total_value * 100, 1), "% of total value )")
print("SKUs with stock but no sales:", no_sales_but_stock.sum())
print()
print("Overstock value by category:")
print(overstock.groupby("Category")["Inventory_Value"].sum().sort_values(ascending=False).round(2))

overstock.to_csv(output_folder + "/overstock_skus.csv", index=False)

Overstocked SKUs: 291 ( 24.2 % of SKUs )
Value tied up   : 465561939.52 ( 63.6 % of total value )
SKUs with stock but no sales: 47

Overstock value by category:
Category
Electronics      163,684,818.73
Home & Kitchen   131,526,160.31
Sports            84,788,122.86
Apparel           59,865,093.37
Grocery           20,528,101.86
Stationery         5,169,642.39
Name: Inventory_Value, dtype: float64


## Task 11 - Inventory value by warehouse location

In [13]:
warehouse_table = df.groupby("Warehouse_Location")[["Inventory_Value"]].sum()
warehouse_table["SKU_Count"] = df.groupby("Warehouse_Location")["SKU_ID"].count()
warehouse_table["Share_Percent"] = warehouse_table["Inventory_Value"] / total_value * 100
warehouse_table = warehouse_table.sort_values("Inventory_Value", ascending=False)

print(warehouse_table.round(1))

# Extra: share of SKUs at risk in each warehouse (used later in recommendations)
warehouse_risk_percent = df.groupby("Warehouse_Location")["Stockout_Risk"].mean() * 100
print()
print("Percent of SKUs at risk in each warehouse:")
print(warehouse_risk_percent.round(1))

                    Inventory_Value  SKU_Count  Share_Percent
Warehouse_Location                                           
Delhi-NCR Hub        216,189,058.40        335          29.50
Mumbai Hub           201,573,372.30        297          27.50
Bengaluru Hub        135,810,243.60        221          18.60
Hyderabad Hub        110,135,170.00        184          15.10
Kolkata Hub           68,062,469.90        163           9.30

Percent of SKUs at risk in each warehouse:
Warehouse_Location
Bengaluru Hub   20.80
Delhi-NCR Hub   20.60
Hyderabad Hub   18.50
Kolkata Hub     17.80
Mumbai Hub      18.50
Name: Stockout_Risk, dtype: float64


## Task 12 - NumPy statistics for supplier lead times

In [14]:
lead_times = df["Lead_Time_Days"].values      # NumPy array

print("Mean lead time   :", round(np.mean(lead_times), 2), "days")
print("Median lead time :", round(np.median(lead_times), 2), "days")
print("Std deviation    :", round(np.std(lead_times), 2), "days")

# Same idea for each supplier, using a loop and a dictionary
supplier_mean_lead = {}
for supplier_name in df["Supplier"].unique():
    values = df.loc[df["Supplier"] == supplier_name, "Lead_Time_Days"].values
    supplier_mean_lead[supplier_name] = round(np.mean(values), 2)

print()
print("Mean lead time per supplier:")
for name in supplier_mean_lead:
    print(" ", name, ":", supplier_mean_lead[name])

Mean lead time   : 10.07 days
Median lead time : 8.0 days
Std deviation    : 5.87 days

Mean lead time per supplier:
  FastTrack Goods : 5.75
  Apex Traders : 6.49
  EastGate Imports : 18.23
  GreenField Co : 14.64
  CityWave Pvt Ltd : 12.07
  BlueLine Supplies : 8.87
  Delta Components : 6.93


## Task 13 - Unusually long lead times (IQR method)
* Q1 = 25th percentile, Q3 = 75th percentile, IQR = Q3 - Q1
* Upper limit = Q3 + 1.5 x IQR. A lead time above this is an outlier.

In [15]:
q1 = np.percentile(lead_times, 25)
q3 = np.percentile(lead_times, 75)
iqr = q3 - q1
lower_limit = q1 - 1.5 * iqr
upper_limit = q3 + 1.5 * iqr

print("Q1:", q1, " Q3:", q3, " IQR:", iqr)
print("Lower limit:", lower_limit, " Upper limit:", upper_limit)

long_lead = df[df["Lead_Time_Days"] > upper_limit]
long_lead = long_lead.sort_values("Lead_Time_Days", ascending=False)

print()
print("SKUs with unusually long lead time:", len(long_lead))
print(long_lead[["SKU_ID", "Product_Name", "Supplier", "Lead_Time_Days"]])
print()
print("Count by supplier:")
print(long_lead["Supplier"].value_counts())

long_lead.to_csv(output_folder + "/lead_time_outliers.csv", index=False)

Q1: 6.0  Q3: 13.0  IQR: 7.0
Lower limit: -4.5  Upper limit: 23.5

SKUs with unusually long lead time: 32
       SKU_ID       Product_Name           Supplier  Lead_Time_Days
345   SKU2132  Bluetooth Speaker       Apex Traders           46.00
949   SKU1539          Lunch Box       Apex Traders           46.00
990   SKU1967  Bluetooth Speaker      GreenField Co           45.00
105   SKU1884      Skipping Rope      GreenField Co           45.00
816   SKU1204    Resistance Band      GreenField Co           43.00
818   SKU1596        Track Pants    FastTrack Goods           43.00
89    SKU1972           Football  BlueLine Supplies           42.00
875   SKU1090           Tea Pack   EastGate Imports           42.00
226   SKU1666        Storage Jar    FastTrack Goods           41.00
307   SKU1978        Gel Pen Box    FastTrack Goods           39.00
10    SKU1627     Desk Organiser    FastTrack Goods           35.00
1107  SKU2004           Yoga Mat   EastGate Imports           34.00
1195  SKU20

## Task 14 - Stock-status label (Critical / Low / Optimal / Overstock)
`np.select` checks the rules **in order** and uses the first one that is true.

| Order | Status | Rule |
|---|---|---|
| 1 | **Critical** | stock is at or below the reorder level **and** it will run out before a new order arrives (Days_Of_Stock <= Lead_Time_Days) |
| 2 | **Low** | stock is at or below the reorder level (but not Critical) |
| 3 | **Overstock** | more than 90 days of stock, or stock with no sales |
| 4 | **Optimal** | everything else |

In [16]:
rule_critical = (df["Stockout_Risk"] == True) & (df["Days_Of_Stock"] <= df["Lead_Time_Days"])
rule_low = df["Stockout_Risk"] == True
rule_overstock = df["Overstock_Flag"] == True

df["Stock_Status"] = np.select(
    [rule_critical, rule_low, rule_overstock],
    ["Critical", "Low", "Overstock"],
    default="Optimal"
)

status_counts = df["Stock_Status"].value_counts()
status_value = df.groupby("Stock_Status")["Inventory_Value"].sum()

print("Number of SKUs in each status:")
print(status_counts)
print()
print("Inventory value in each status:")
print(status_value.round(2))
print()
print("Status by category:")
print(pd.crosstab(df["Category"], df["Stock_Status"]))

Number of SKUs in each status:
Stock_Status
Optimal      676
Overstock    291
Critical     172
Low           61
Name: count, dtype: int64

Inventory value in each status:
Stock_Status
Critical      9,426,510.06
Low           7,078,346.95
Optimal     249,703,517.61
Overstock   465,561,939.52
Name: Inventory_Value, dtype: float64

Status by category:
Stock_Status    Critical  Low  Optimal  Overstock
Category                                         
Apparel               27   13      116         53
Electronics           42   10      123         64
Grocery               29    9      108         41
Home & Kitchen        37   16      135         64
Sports                19    6      100         35
Stationery            18    7       94         34


## A reusable function (extra)
This function returns the total inventory value of one category, so we do not repeat the same calculation.

In [17]:
def category_value(category_name):
    """Return the total inventory value of one category."""
    rows = df[df["Category"] == category_name]
    return rows["Inventory_Value"].sum()

for category_name in df["Category"].unique():
    print(category_name, ":", round(category_value(category_name), 2))

Grocery : 30591314.24
Apparel : 88478504.52
Electronics : 253762710.0
Home & Kitchen : 197681756.37
Sports : 149656180.65
Stationery : 11599848.36


## Task 15 - Five recommendations
The numbers in each sentence come from the results above (they are printed from variables, not typed by hand). These are patterns in a synthetic dataset, not proof of cause and effect.

In [18]:
# --- Numbers needed ---
critical_count = (df["Stock_Status"] == "Critical").sum()
riskiest_category = risk_table.index[0]
riskiest_percent = risk_table["Risk_Percent"].iloc[0]

overstock_percent = overstock_value / total_value * 100
overstock_by_category = overstock.groupby("Category")["Inventory_Value"].sum().sort_values(ascending=False)
top_overstock_category = overstock_by_category.index[0]

best_supplier = supplier_table.index[0]
worst_supplier = supplier_table.index[-1]
worst_lead = supplier_table.loc[worst_supplier, "Lead_Time_Days"]
worst_defect = supplier_table.loc[worst_supplier, "Defect_Return_Rate_Pct"]
best_lead = supplier_table.loc[best_supplier, "Lead_Time_Days"]

# Reorder levels that are smaller than the sales expected during the lead time
demand_in_lead_time = df["Avg_Daily_Sales"] * df["Lead_Time_Days"]
weak_reorder_count = (df["Reorder_Level"] < demand_in_lead_time).sum()

warehouse_most_risk = warehouse_risk_percent.idxmax()
warehouse_most_overstock = overstock.groupby("Warehouse_Location")["Inventory_Value"].sum().idxmax()

# --- Recommendations ---
print("1. Reorder critical items first.")
print("  ", critical_count, "SKUs are below their reorder level and will run out before a new order arrives.")
print("   The category", riskiest_category, "has the highest share of SKUs at risk (", round(riskiest_percent, 1), "% ).")
print()
print("2. Free the money locked in slow stock.")
print("  ", len(overstock), "SKUs have more than", overstock_days, "days of stock and hold", round(overstock_percent, 1), "% of inventory value.")
print("   Pause buying and run offers, starting with the", top_overstock_category, "category.")
print("  ", no_sales_but_stock.sum(), "SKUs have stock but no sales - consider clearing or stopping them.")
print()
print("3. Manage supplier risk.")
print("  ", worst_supplier, "ranks last (average lead time", round(worst_lead, 1), "days, defect/return rate", round(worst_defect, 1), "%).")
print("  ", best_supplier, "ranks first (average lead time", round(best_lead, 1), "days).")
print("   Move some orders to better suppliers, and keep extra safety stock for slow ones.")
print("  ", len(long_lead), "SKUs have unusually long lead times (above", upper_limit, "days) and need a backup supplier.")
print()
print("4. Watch the highest-value items closely.")
print("   The top 10 SKUs hold", round(top10_percent, 1), "% of the total inventory value. Check them every week.")
print()
print("5. Fix reorder levels and balance the warehouses.")
print("  ", weak_reorder_count, "SKUs have a reorder level lower than the sales expected during the lead time.")
print("   Recalculate: reorder level = daily sales x lead time + safety stock.")
print("  ", warehouse_most_overstock, "has the most overstock value; ", warehouse_most_risk, "has the highest share of SKUs at risk.")
print("   If they are different hubs, consider moving stock between them before buying more.")

1. Reorder critical items first.
   172 SKUs are below their reorder level and will run out before a new order arrives.
   The category Electronics has the highest share of SKUs at risk ( 21.8 % ).

2. Free the money locked in slow stock.
   291 SKUs have more than 90 days of stock and hold 63.6 % of inventory value.
   Pause buying and run offers, starting with the Electronics category.
   47 SKUs have stock but no sales - consider clearing or stopping them.

3. Manage supplier risk.
   EastGate Imports ranks last (average lead time 18.2 days, defect/return rate 4.8 %).
   FastTrack Goods ranks first (average lead time 5.7 days).
   Move some orders to better suppliers, and keep extra safety stock for slow ones.
   32 SKUs have unusually long lead times (above 23.5 days) and need a backup supplier.

4. Watch the highest-value items closely.
   The top 10 SKUs hold 14.4 % of the total inventory value. Check them every week.

5. Fix reorder levels and balance the warehouses.
   14 SKUs 

## Save the cleaned data and a short summary file

In [19]:
df.to_csv(root + "/data/cleaned_inventory.csv", index=False)

summary = open(output_folder + "/analysis_summary.txt", "w")
summary.write("Warehouse Inventory & Stockout Risk - Summary\n")
summary.write("SKUs analysed: " + str(len(df)) + "\n")
summary.write("Total inventory value: " + str(round(total_value, 2)) + "\n")
summary.write("Stockout-risk SKUs: " + str(risk_count) + "\n")
summary.write("Overstocked SKUs: " + str(len(overstock)) + "\n")
summary.write("Critical SKUs: " + str(critical_count) + "\n")
summary.close()

print("Saved files:", os.listdir(output_folder))

Saved files: ['analysis_summary.txt', 'lead_time_outliers.csv', 'overstock_skus.csv', 'stockout_risk_skus.csv', 'supplier_ranking.csv']


## Assumptions and limitations
* The data is synthetic. Real warehouses have seasons, offers and supplier contracts that are not in this data.
* Average daily sales is treated as a fixed number, so "days of stock" is only an estimate.
* Filled-in lead times and unit costs are estimates (True/False columns mark them).
* The supplier rank gives lead time and defect rate equal weight. Other weights could change the order.
* All results are patterns in this dataset, not proof that one thing causes another.